# AegisFlow — Phase 1 EDA: CIC-IDS2017

Run `python scripts/prepare_data.py --dataset cic_ids2017` first (or with `--sample-size 200000` for a quick pass) so that `data/interim/cic_ids2017.parquet` exists.

Everything below reads that real, ingested data — nothing here is synthetic.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
from aegisflow.config import load_config

cfg = load_config(root=__import__('pathlib').Path('..').resolve())
df = pd.read_parquet(cfg.path(cfg.config.paths.data_interim) / 'cic_ids2017.parquet')
df.shape

## 1. Basic shape and time range

In [ ]:
print('rows:', len(df))
print('unique source hosts:', df['source_ip'].nunique())
print('unique destination hosts:', df['destination_ip'].nunique())
print('time range:', df['timestamp'].min(), '->', df['timestamp'].max())

## 2. Label pipeline: dataset_label -> normalized_attack_class -> attack_stage

Remember: `attack_stage` is an AegisFlow-inferred proxy (see `configs/stages.yaml`), not a field the dataset itself provides.

In [ ]:
pd.crosstab(df['dataset_label'], df['attack_stage'])

In [ ]:
df['label_confidence'].value_counts()

## 3. Class imbalance

In [ ]:
counts = df['attack_stage'].value_counts()
ax = counts.plot(kind='bar', figsize=(8,4), color='#1F3864', logy=True)
ax.set_ylabel('flow count (log scale)')
ax.set_title('CIC-IDS2017: attack_stage distribution')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()

## 4. Traffic volume over time (benign vs attack)

This is what justifies time-based (not random) train/val/test splits: attack traffic is concentrated in specific windows per day.

In [ ]:
ts = df.set_index('timestamp').copy()
ts['is_attack'] = ts['attack_stage'].ne('Benign')
hourly = ts.groupby([pd.Grouper(freq='1h'), 'is_attack']).size().unstack(fill_value=0)
hourly.plot(figsize=(10,4), color={False: '#6FAE8C', True: '#E8710C'})
plt.ylabel('flows / hour')
plt.title('Traffic volume over time')
plt.tight_layout()

## 5. Missing-value fractions by column

Confirms which canonical features CIC-IDS2017 genuinely supplies vs. which are structurally unavailable (e.g. `ttl_mean`) — cross-check against `python -m aegisflow feature-registry`.

In [ ]:
df.isna().mean().sort_values(ascending=False).to_frame('missing_fraction')

## 6. Per-host flow-count distribution

Sanity check for the windowing step in Phase 2: how many flows does a typical source host generate, and how skewed is it?

In [ ]:
per_host = df.groupby('source_ip').size().sort_values(ascending=False)
print(per_host.describe())
per_host.head(15)